# Solutions — UDFs and Pandas UDFs (Hard 09)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.functions import pandas_udf, udf
from pyspark.sql import Window
import pandas as pd

trips        = spark.table("samples.nyctaxi.trips")
transactions = spark.table("samples.bakehouse.sales_transactions")
customers    = spark.table("samples.bakehouse.sales_customers")


## Solution 1 — Basic Python UDF — Fare Category

In [ ]:
@udf(T.StringType())
def fare_category(amount):
    if amount is None:       return None
    if amount < 5:           return "low"
    elif amount < 15:        return "medium"
    elif amount < 30:        return "high"
    else:                    return "premium"

result_1 = (
    trips
    .withColumn("fare_category", fare_category("fare_amount"))
    .groupBy("fare_category")
    .agg(F.count("*").alias("trip_count"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
    .orderBy("fare_category")
)


## Solution 2 — UDF with Multiple Inputs — Formatted Name

In [ ]:
@udf(T.StringType())
def formatted_name(first, last):
    if first is None or last is None:
        return None
    return f"{last.upper()}, {first.title()}"

result_2 = (
    customers
    .withColumn("formatted_name", formatted_name("firstName", "lastName"))
    .select("customerID", "firstName", "lastName", "formatted_name")
    .limit(100)
)


## Solution 3 — UDF for Card Number Masking

In [ ]:
@udf(T.StringType())
def mask_card(card_num):
    if card_num is None:
        return None
    s = str(card_num)
    return "****-****-****-" + s[-4:]

result_3 = (
    transactions
    .withColumn("masked_card", mask_card(F.col("cardNumber").cast("string")))
    .select("transactionID", "cardNumber", "masked_card")
    .limit(100)
)


## Solution 4 — pandas_udf (Scalar) — Z-score of Fare Amount

In [ ]:
@pandas_udf(T.DoubleType())
def zscore_fare(fares: pd.Series) -> pd.Series:
    mean = fares.mean()
    std  = fares.std()
    if std == 0:
        return pd.Series([0.0] * len(fares))
    return (fares - mean) / std

result_4 = (
    trips
    .withColumn("fare_zscore", F.round(zscore_fare("fare_amount"), 4))
    .select("tpep_pickup_datetime", "fare_amount", "fare_zscore")
    .filter(F.col("fare_amount") > 0)
)


## Solution 5 — pandas_udf for String Normalisation

In [ ]:
@pandas_udf(T.StringType())
def normalise_product(names: pd.Series) -> pd.Series:
    return names.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)

result_5 = (
    transactions
    .withColumn("product_normalised", normalise_product("product"))
    .select("transactionID", "product", "product_normalised")
    .distinct()
)


## Solution 6 — Register UDF for SQL Use

In [ ]:
@udf(T.StringType())
def payment_label(method):
    labels = {"CASH": "💵 Cash", "CARD": "💳 Card", "CHEQUE": "📝 Cheque"}
    return labels.get(method, method) if method else None

spark.udf.register("payment_label_sql", payment_label)

result_6 = spark.sql("""
    SELECT paymentMethod,
           payment_label_sql(paymentMethod) AS payment_label,
           COUNT(*) AS transaction_count
    FROM   sales_transactions_view
    GROUP BY paymentMethod
    ORDER BY transaction_count DESC
""")


## Solution 7 — UDF Performance Comparison — Built-ins Win

In [ ]:
# Demonstrate that built-in functions are faster than equivalent UDFs
# Show both produce the same result

@udf(T.StringType())
def upper_udf(s):
    return s.upper() if s else None

result_7 = spark.createDataFrame(
    [("udf",     transactions.withColumn("p", upper_udf("product")).select("p").limit(1000).count()),
     ("builtin", transactions.withColumn("p", F.upper("product")).select("p").limit(1000).count())],
    ["approach", "row_count"]
)


## Solution 8 — Grouped Map with applyInPandas

In [ ]:
def normalise_by_zip(pdf: pd.DataFrame) -> pd.DataFrame:
    lo = pdf["fare_amount"].min()
    hi = pdf["fare_amount"].max()
    pdf["normalised_fare"] = ((pdf["fare_amount"] - lo) / (hi - lo)).round(4) if hi != lo else 0.0
    return pdf[["pickup_zip", "fare_amount", "normalised_fare"]]

schema = "pickup_zip INT, fare_amount DOUBLE, normalised_fare DOUBLE"

result_8 = (
    trips
    .filter(F.col("fare_amount") > 0)
    .groupBy("pickup_zip")
    .applyInPandas(normalise_by_zip, schema=schema)
)
